# Troubleshooting API Not Responding
This notebook demonstrates how to diagnose and fix issues when an external API stops responding. We'll simulate a generic HTTP request scenario and implement robust handling, including retries, logging, and mock testing.

In [1]:
# Section 1: Import Libraries
import requests
import time
import logging
from requests.exceptions import Timeout, RequestException

logging.basicConfig(level=logging.DEBUG)

In [2]:
# Section 2: Define API Request Function
def make_api_request(url, params=None, timeout=5):
    try:
        response = requests.get(url, params=params, timeout=timeout)
        return response
    except Timeout:
        logging.error("Request timed out")
        raise
    except RequestException as e:
        logging.error(f"Request failed: {e}")
        raise

In [3]:
# Section 3: Handle Timeouts and Retries
def request_with_retry(url, params=None, retries=3, backoff=1):
    for attempt in range(1, retries + 1):
        try:
            resp = make_api_request(url, params=params)
            return resp
        except Timeout:
            logging.warning(f"Timeout on attempt {attempt}, retrying after {backoff}s")
            time.sleep(backoff)
            backoff *= 2
    raise Timeout("Max retries exceeded")

In [4]:
# Section 4: Check Response Status and Errors
def check_response(resp):
    try:
        resp.raise_for_status()
    except RequestException as e:
        logging.error(f"HTTP error: {e} - {resp.text}")
        raise
    try:
        data = resp.json()
    except ValueError:
        logging.error("Invalid JSON in response")
        raise
    return data

In [5]:
# Section 5: Implement Logging and Debugging
logging.getLogger().setLevel(logging.DEBUG)

def debug_request(url):
    logging.debug(f"Sending request to {url}")
    resp = request_with_retry(url)
    logging.debug(f"Received status {resp.status_code}")
    logging.debug(f"Response body: {resp.text}")
    return resp

In [6]:
# Section 6: Test with Mock Server
# We'll use requests_mock if available; otherwise, demonstrate with a simple flask server.

try:
    import requests_mock
    
    with requests_mock.Mocker() as m:
        m.get('https://api.example.com/data', json={'key': 'value'})
        resp = debug_request('https://api.example.com/data')
        print('Mock response data:', resp.json())
except ImportError:
    print('requests_mock not installed; manually start a local server to test')

requests_mock not installed; manually start a local server to test


In [8]:
# Extra: Inspect google.genai module
import google.genai as genai_new
print('genai_new:', genai_new)
print('dir:', [name for name in dir(genai_new) if not name.startswith('_')])


genai_new: <module 'google.genai' from 'c:\\Users\\manip\\Desktop\\Agentic AI\\.venv\\Lib\\site-packages\\google\\genai\\__init__.py'>
dir: ['Client', 'batches', 'caches', 'chats', 'client', 'documents', 'errors', 'file_search_stores', 'files', 'interactions', 'live', 'live_music', 'models', 'operations', 'pagers', 'tokens', 'tunings', 'types', 'version']


In [10]:
# Inspect Client class
from google.genai import Client
print(Client)
print(dir(Client))


<class 'google.genai.client.Client'>
['__class__', '__del__', '__delattr__', '__dict__', '__dir__', '__doc__', '__enter__', '__eq__', '__exit__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_get_api_client', '_has_nextgen_client', '_nextgen_client', 'aio', 'auth_tokens', 'batches', 'caches', 'chats', 'close', 'file_search_stores', 'files', 'interactions', 'models', 'operations', 'tunings', 'vertexai']


In [11]:
# Inspect chat create method signature
client = Client()
import inspect
print(inspect.signature(client.chats.create))
print(client.chats.create.__doc__)


(*, model: str, config: Union[google.genai.types.GenerateContentConfig, google.genai.types.GenerateContentConfigDict, NoneType] = None, history: Optional[list[Union[google.genai.types.Content, google.genai.types.ContentDict]]] = None) -> google.genai.chats.Chat
Creates a new chat session.

Args:
  model: The model to use for the chat.
  config: The configuration to use for the generate content request.
  history: The history to use for the chat.

Returns:
  A new chat session.



In [12]:
# Explore chat session returned object
chat = client.chats.create(model="gemini-1.5-flash-latest")
print(chat)
print(dir(chat))


['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_comprehensive_history', '_config', '_curated_history', '_model', '_modules', 'get_history', 'record_history', 'send_message', 'send_message_stream']


In [14]:
print('send_message signature:', inspect.signature(chat.send_message))
print(chat.send_message.__doc__)


send_message signature: (message: Union[list[Union[str, google.genai.types.File, google.genai.types.FileDict, google.genai.types.Part, google.genai.types.PartDict]], str, google.genai.types.File, google.genai.types.FileDict, google.genai.types.Part, google.genai.types.PartDict], config: Union[google.genai.types.GenerateContentConfig, google.genai.types.GenerateContentConfigDict, NoneType] = None) -> google.genai.types.GenerateContentResponse
Sends the conversation history with the additional message and returns the model's response.

Args:
  message: The message to send to the model.
  config:  Optional config to override the default Chat config for this
    request.

Returns:
  The model's response.

Usage:

.. code-block:: python

  chat = client.chats.create(model='gemini-2.0-flash')
  response = chat.send_message('tell me a story')



In [20]:
print('record_history signature:', inspect.signature(chat.record_history))
print(chat.record_history.__doc__)


record_history signature: (user_input: google.genai.types.Content, model_output: list[google.genai.types.Content], automatic_function_calling_history: list[google.genai.types.Content], is_valid: bool) -> None
Records the chat history.

Maintaining both comprehensive and curated histories.

Args:
  user_input: The user's input content.
  model_output: A list of `Content` from the model's response. This can be
    an empty list if the model produced no output.
  automatic_function_calling_history: A list of `Content` representing the
    history of automatic function calls, including the user input as the
    first entry.
  is_valid: A boolean flag indicating whether the current model output is
    considered valid.



In [15]:
# Inspect responses
print('responses attribute:', dir(client.responses))
import inspect
print('responses.create signature', inspect.signature(client.responses.create))
print(client.responses.create.__doc__)


AttributeError: 'Client' object has no attribute 'responses'

In [17]:
print('Client init signature:', inspect.signature(Client))


Client init signature: (*, vertexai: Optional[bool] = None, api_key: Optional[str] = None, credentials: Optional[google.auth.credentials.Credentials] = None, project: Optional[str] = None, location: Optional[str] = None, debug_config: Optional[google.genai.client.DebugConfig] = None, http_options: Union[google.genai.types.HttpOptions, google.genai.types.HttpOptionsDict, NoneType] = None)


In [18]:
# Attempt a live request (may fail if key invalid or network blocked)
from dotenv import load_dotenv
load_dotenv()
import os
client = Client(api_key=os.getenv("GEMINI_API_KEY"))
chat = client.chats.create(model="gemini-1.5-flash-latest")
try:
    resp = chat.send_message("Hello from notebook")
    print('response object:', resp)
    # try to access text
    # resp might be a GenerateContentResponse
    print('response output:', resp.output)
except Exception as err:
    print('error during API call:', err)


INFO:google_genai.models:AFC is enabled with max remote calls: 10.
DEBUG:httpcore.connection:connect_tcp.started host='generativelanguage.googleapis.com' port=443 local_address=None timeout=None socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000026DE1B24440>
DEBUG:httpcore.connection:start_tls.started ssl_context=<ssl.SSLContext object at 0x0000026DDF928CD0> server_hostname='generativelanguage.googleapis.com' timeout=None
DEBUG:httpcore.connection:start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000026DE1B39450>
DEBUG:httpcore.http11:send_request_headers.started request=<Request [b'POST']>
DEBUG:httpcore.http11:send_request_headers.complete
DEBUG:httpcore.http11:send_request_body.started request=<Request [b'POST']>
DEBUG:httpcore.http11:send_request_body.complete
DEBUG:httpcore.http11:receive_response_headers.started request=<Request [b'POST']>
DEBUG:httpcore.http11:receive

error during API call: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash-latest is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}


In [19]:
# List available models
models = client.models.list()
print('Available models count:', len(models))
for m in models[:10]:
    print(m.name)


DEBUG:httpcore.connection:close.started
DEBUG:httpcore.connection:close.complete
DEBUG:httpcore.connection:connect_tcp.started host='generativelanguage.googleapis.com' port=443 local_address=None timeout=None socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000026DE149A210>
DEBUG:httpcore.connection:start_tls.started ssl_context=<ssl.SSLContext object at 0x0000026DDF928CD0> server_hostname='generativelanguage.googleapis.com' timeout=None
DEBUG:httpcore.connection:start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000026DE134EC40>
DEBUG:httpcore.http11:send_request_headers.started request=<Request [b'GET']>
DEBUG:httpcore.http11:send_request_headers.complete
DEBUG:httpcore.http11:send_request_body.started request=<Request [b'GET']>
DEBUG:httpcore.http11:send_request_body.complete
DEBUG:httpcore.http11:receive_response_headers.started request=<Request [b'GET']>
DEBUG:httpcore.htt

Available models count: 43
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it


In [23]:
# Try a successful chat with valid model name
chat = client.chats.create(model="gemini-2.5-flash")
resp = chat.send_message("Hello, please respond with a short greeting.")
print(resp)
print('output attr type', type(resp.output))
print('output contents', resp.output)

# sometimes output is a list of Items with content list
if resp.output:
    for item in resp.output:
        print('item type', item.type, 'content', item.content)
        if item.content:
            for c in item.content:
                print('content type', c.type, 'text', getattr(c,'text',None))


INFO:google_genai.models:AFC is enabled with max remote calls: 10.
DEBUG:httpcore.connection:close.started
DEBUG:httpcore.connection:close.complete
DEBUG:httpcore.connection:connect_tcp.started host='generativelanguage.googleapis.com' port=443 local_address=None timeout=None socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000026DE1512CF0>
DEBUG:httpcore.connection:start_tls.started ssl_context=<ssl.SSLContext object at 0x0000026DDF928CD0> server_hostname='generativelanguage.googleapis.com' timeout=None
DEBUG:httpcore.connection:start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000026DE1512E00>
DEBUG:httpcore.http11:send_request_headers.started request=<Request [b'POST']>
DEBUG:httpcore.http11:send_request_headers.complete
DEBUG:httpcore.http11:send_request_body.started request=<Request [b'POST']>
DEBUG:httpcore.http11:send_request_body.complete
DEBUG:httpcore.http11:receive_r

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 14.017285944s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'},

In [22]:
# Inspect response class attributes
print('dir(resp)=', [a for a in dir(resp) if not a.startswith('_')])


NameError: name 'resp' is not defined